# M10 OBSO + C-OBSO — run partido a partido (resumable)

Procesa los 64 partidos WC22 a **25 Hz full quality**, uno a uno.

Cachea cada partido por separado en `data/parquet/derived/offball/per_match/{match_id}.parquet`.
Se puede **interrumpir y retomar**: al re-ejecutar el NB salta los partidos ya cacheados.

Cuando los 64 esten cacheados, las celdas finales agregan `per_minute.parquet` y `per_shock_window.parquet`.

In [1]:
# Setup
import sys, gc, time
from pathlib import Path
import polars as pl

_REPO = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(_REPO / 'src'))

from M01_loader_pff import list_event_match_ids
import M10_offball as m10

PER_MATCH_DIR = _REPO / 'data' / 'parquet' / 'derived' / 'offball' / 'per_match'
PER_MATCH_DIR.mkdir(parents=True, exist_ok=True)
print(f'cache dir: {PER_MATCH_DIR}')

cache dir: /home/joriol/TFM/data/parquet/derived/offball/per_match


In [2]:
# xG grid (cached, ~5s)
xg_grid = m10.build_xg_grid(cache=True)
print(f'xG grid {xg_grid.shape}: min={xg_grid.min():.4f} max={xg_grid.max():.4f}')

xG grid (10, 7): min=0.0050 max=0.1609


In [3]:
# Loop partido a partido. Salta los ya cacheados.
# Tiempo por partido a 25 Hz: ~10-15 min.  64 partidos = ~12-15h total.
# Interrumpir es seguro: re-ejecutar el NB retoma desde donde quedo.

match_ids = list_event_match_ids()
done   = {int(p.stem) for p in PER_MATCH_DIR.glob('*.parquet')}
todo   = [m for m in match_ids if m not in done]
print(f'progreso: {len(done)}/{len(match_ids)} cacheados, {len(todo)} pendientes')

for i, mid in enumerate(todo):
    t0 = time.time()
    try:
        df = m10.compute_obso_match(mid, xg_grid)
    except Exception as e:
        print(f'  [{i+1}/{len(todo)}] match {mid} FAIL: {e}')
        continue
    out = PER_MATCH_DIR / f'{mid}.parquet'
    df.write_parquet(out, compression='snappy', statistics=True)
    elapsed = time.time() - t0
    total_done = len(done) + i + 1
    print(f'  [{total_done}/{len(match_ids)}] match {mid}: {df.height:>5} rows en {elapsed:>6.1f}s -> {out.name}', flush=True)
    del df
    gc.collect()

print(f'\ncompletados: {len(list(PER_MATCH_DIR.glob("*.parquet")))}/{len(match_ids)}')

progreso: 0/64 cacheados, 64 pendientes
  [1/64] match 3812:  1959 rows en  199.1s -> 3812.parquet
  [2/64] match 3813:  1870 rows en  201.2s -> 3813.parquet
  [3/64] match 3814:  1826 rows en  200.8s -> 3814.parquet
  [4/64] match 3815:  1991 rows en  221.4s -> 3815.parquet
  [5/64] match 3816:  1925 rows en  201.3s -> 3816.parquet
  [6/64] match 3817:  2003 rows en  209.9s -> 3817.parquet
  [7/64] match 3818:  1903 rows en  199.5s -> 3818.parquet
  [8/64] match 3819:  1892 rows en  263.3s -> 3819.parquet
  [9/64] match 3820:  1948 rows en  229.1s -> 3820.parquet
  [10/64] match 3821:  1881 rows en  208.7s -> 3821.parquet
  [11/64] match 3822:  1892 rows en  268.6s -> 3822.parquet
  [12/64] match 3823:  1941 rows en  220.9s -> 3823.parquet
  [13/64] match 3824:  2024 rows en  249.1s -> 3824.parquet
  [14/64] match 3825:  1903 rows en  210.5s -> 3825.parquet
  [15/64] match 3826:  1837 rows en  224.0s -> 3826.parquet
  [16/64] match 3827:  1936 rows en  217.6s -> 3827.parquet
  [17/64]

## Agregacion final

Solo correr cuando los 64 partidos esten cacheados. Concatena los 64 parquets y aplica la agregacion shock-window.

In [4]:
# Concat de los 64 per-match -> per_minute.parquet
files = sorted(PER_MATCH_DIR.glob('*.parquet'))
assert len(files) == 64, f'faltan {64 - len(files)} partidos por procesar'

big = pl.concat([pl.read_parquet(f) for f in files])
out_pm = _REPO / 'data' / 'parquet' / 'derived' / 'offball' / 'per_minute.parquet'
big.write_parquet(out_pm, compression='snappy', statistics=True)
print(f'per_minute: {big.height:,} filas, {big["pff_match_id"].n_unique()} matches -> {out_pm}')
print(big.head(3))

per_minute: 124,762 filas, 64 matches -> /home/joriol/TFM/data/parquet/derived/offball/per_minute.parquet
shape: (3, 7)
┌──────────────┬───────────┬────────┬───────────┬──────────┬─────────────┬──────────────────┐
│ pff_match_id ┆ player_id ┆ minute ┆ obso_mean ┆ obso_max ┆ c_obso_mean ┆ attacking_frames │
│ ---          ┆ ---       ┆ ---    ┆ ---       ┆ ---      ┆ ---         ┆ ---              │
│ i64          ┆ i64       ┆ i64    ┆ f64       ┆ f64      ┆ f64         ┆ u32              │
╞══════════════╪═══════════╪════════╪═══════════╪══════════╪═════════════╪══════════════════╡
│ 10502        ┆ 13121     ┆ 97     ┆ 0.032398  ┆ 0.09999  ┆ 0.00023     ┆ 571              │
│ 10502        ┆ 8343      ┆ 31     ┆ 0.0       ┆ 0.0      ┆ null        ┆ 1                │
│ 10502        ┆ 802       ┆ 30     ┆ 0.001677  ┆ 0.057034 ┆ null        ┆ 34               │
└──────────────┴───────────┴────────┴───────────┴──────────┴─────────────┴──────────────────┘


In [5]:
# Per-shock-window aggregation usando la funcion de M10
import importlib; importlib.reload(m10)
per_shock = m10.aggregate_per_shock_window(cache=False)
out_ps = _REPO / 'data' / 'parquet' / 'derived' / 'offball' / 'per_shock_window.parquet'
per_shock.write_parquet(out_ps, compression='snappy', statistics=True)
print(f'per_shock_window: {per_shock.height} filas -> {out_ps}')

# Sanity delta por shock_type
summary = per_shock.group_by('shock_type').agg([
    pl.col('obso_pre').mean().alias('obso_pre'),
    pl.col('obso_post').mean().alias('obso_post'),
    (pl.col('obso_post') - pl.col('obso_pre')).mean().alias('delta_obso'),
    pl.col('c_obso_pre').mean().alias('c_obso_pre'),
    pl.col('c_obso_post').mean().alias('c_obso_post'),
])
print(summary)

per_shock_window: 3788 filas -> /home/joriol/TFM/data/parquet/derived/offball/per_shock_window.parquet
shape: (2, 6)
┌──────────────┬──────────┬───────────┬────────────┬────────────┬─────────────┐
│ shock_type   ┆ obso_pre ┆ obso_post ┆ delta_obso ┆ c_obso_pre ┆ c_obso_post │
│ ---          ┆ ---      ┆ ---       ┆ ---        ┆ ---        ┆ ---         │
│ str          ┆ f64      ┆ f64       ┆ f64        ┆ f64        ┆ f64         │
╞══════════════╪══════════╪═══════════╪════════════╪════════════╪═════════════╡
│ GOAL_FOR     ┆ 0.006476 ┆ 0.006801  ┆ 0.000288   ┆ -0.000495  ┆ -0.000563   │
│ GOAL_AGAINST ┆ 0.005335 ┆ 0.005704  ┆ 0.000268   ┆ -0.000438  ┆ -0.000409   │
└──────────────┴──────────┴───────────┴────────────┴────────────┴─────────────┘
